In [1]:
dataset = []
with open('cat-facts.txt', 'r') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')



Loaded 195 entries


In [2]:
import ollama
print(ollama.list())

models=[Model(model='hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest', modified_at=datetime.datetime(2025, 6, 21, 1, 18, 26, 149851, tzinfo=TzInfo(+05:45)), digest='042cf58aa32fa6ce631caff838fc9efd05313a629530756ccf59308eaa49b8aa', size=807696561, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='1.24B', quantization_level='unknown')), Model(model='hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest', modified_at=datetime.datetime(2025, 6, 21, 1, 17, 41, 481411, tzinfo=TzInfo(+05:45)), digest='98c4eb4a3287679e9d01ee6bd664136c194871e741bb405b082ffa65fed19c95', size=68348847, details=ModelDetails(parent_model='', format='gguf', family='bert', families=['bert'], parameter_size='109M', quantization_level='unknown')), Model(model='llama2:latest', modified_at=datetime.datetime(2025, 6, 21, 0, 26, 1, 846052, tzinfo=TzInfo(+05:45)), digest='78e26419b4469263f75331927a00a0284ef6544c1975b826b15abdaef17bb962', size=3826793677, details=ModelDet

In [3]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

In [4]:
pip show ollama

Name: ollama
Version: 0.5.1
Summary: The official Python client for Ollama.
Home-page: https://ollama.com
Author: 
Author-email: hello@ollama.com
License: 
Location: /home/nhujaw/miniconda3/lib/python3.13/site-packages
Requires: httpx, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

Added chunk 1/195 to the database
Added chunk 2/195 to the database
Added chunk 3/195 to the database
Added chunk 4/195 to the database
Added chunk 5/195 to the database
Added chunk 6/195 to the database
Added chunk 7/195 to the database
Added chunk 8/195 to the database
Added chunk 9/195 to the database
Added chunk 10/195 to the database
Added chunk 11/195 to the database
Added chunk 12/195 to the database
Added chunk 13/195 to the database
Added chunk 14/195 to the database
Added chunk 15/195 to the database
Added chunk 16/195 to the database
Added chunk 17/195 to the database
Added chunk 18/195 to the database
Added chunk 19/195 to the database
Added chunk 20/195 to the database
Added chunk 21/195 to the database
Added chunk 22/195 to the database
Added chunk 23/195 to the database
Added chunk 24/195 to the database
Added chunk 25/195 to the database
Added chunk 26/195 to the database
Added chunk 27/195 to the database
Added chunk 28/195 to the database
Added chunk 29/195 to the dat

In [6]:
#Here is an example function to calculate the cosine similarity between two vectors:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [7]:
# retrieval function:
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]



In [8]:
#Generation function:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

IndexError: list index out of range

In [ ]:
#Using ollama to generate a response:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)